In [261]:
import pandas as pd

observations = pd.read_csv("../data/raw/observations.csv")

observations.shape
observations.info()
observations.head(5)
observations.describe()
observations.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 75343 entries, 0 to 75342
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   DATE         75343 non-null  str  
 1   PATIENT      75343 non-null  str  
 2   ENCOUNTER    72334 non-null  str  
 3   CATEGORY     72334 non-null  str  
 4   CODE         75343 non-null  str  
 5   DESCRIPTION  75343 non-null  str  
 6   VALUE        75343 non-null  str  
 7   UNITS        54279 non-null  str  
 8   TYPE         75343 non-null  str  
dtypes: str(9)
memory usage: 5.2 MB


DATE               0
PATIENT            0
ENCOUNTER       3009
CATEGORY        3009
CODE               0
DESCRIPTION        0
VALUE              0
UNITS          21064
TYPE               0
dtype: int64

In [262]:
observations.columns


Index(['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION',
       'VALUE', 'UNITS', 'TYPE'],
      dtype='str')

### Column Dictionary

| Column | Meaning |
|---|---|
| PATIENT | ... |
| DATE | ... |
| DESCRIPTION | ... |
| VALUE | ... |
| UNITS | ... |
| ENCOUNTER | ... |
| CATEGORY | ... |
| TYPE | ... |

In [263]:
print(len(observations))
print(len(observations["PATIENT"].unique()))
print(observations.groupby("PATIENT").size().sort_values(ascending=False).head(10))
print('the average number of observations per patient:', observations.groupby("PATIENT").size().mean())

75343
108
PATIENT
5d84e6a3-b4bd-63d6-57c5-cada0916490d    13164
7ad140ab-bfae-c3ae-20a3-2244b1c4d0e2     9040
688c8453-ba6f-7dec-03c3-eaa27d6df1a4     5585
01a006ce-6457-50c2-8e0a-fb58fc310a86     3718
8d7f6a31-31ba-da9c-2b57-03ee0f7577a0     3089
c1e9a9fb-45ec-4ee4-c946-4ffc5dfa93ea     2664
53e4891a-9108-67ef-d973-3b6a98404249     1767
9277390b-8a92-52e1-4d99-bb4849bac775     1180
d20a36fc-23ba-8462-bf39-864000fbf25f      929
ba234ff2-cefe-dfee-935a-8ea2378da8c2      828
dtype: int64
the average number of observations per patient: 697.6203703703703


In [264]:
print(observations['DESCRIPTION'].value_counts().head(20))
print(observations['DESCRIPTION'].nunique())

DESCRIPTION
Pain severity - 0-10 verbal numeric rating [Score] - Reported                                                                                 2110
Diastolic Blood Pressure                                                                                                                      1487
Systolic Blood Pressure                                                                                                                       1487
Body Weight                                                                                                                                   1477
Heart rate                                                                                                                                    1471
Respiratory rate                                                                                                                              1471
Body Height                                                                                               

In [265]:
# measurement structure
observations[['DESCRIPTION','UNITS', 'VALUE']].head(10)

,DESCRIPTION,UNITS,VALUE
0,Body Height,cm,175.2
1,Pain severity - 0-10 verbal numeric rating [Sc...,{score},3.0
2,Body Weight,kg,71.1
3,Body mass index (BMI) [Ratio],kg/m2,23.2
4,Diastolic Blood Pressure,mm[Hg],83.0
5,Systolic Blood Pressure,mm[Hg],125.0
6,Heart rate,/min,65.0
7,Respiratory rate,/min,13.0
8,Tobacco smoking status,NaN,Ex-smoker (finding)
9,Within the last year have you been afraid of ...,NaN,No


In [266]:
observations.dtypes
observations['VALUE'].head(10) 
observations['VALUE'].unique()[:50]

numeric_values = pd.to_numeric(
    observations["VALUE"],
    errors="coerce"
)

print(numeric_values.notna().sum())
print(numeric_values.isna().sum())

observations["VALUE_NUMERIC"] = pd.to_numeric(
    observations["VALUE"], errors="coerce"
)

observations[["VALUE_NUMERIC", "VALUE", "DESCRIPTION","UNITS"]].head(10)

observations['VALUE'].value_counts().head(20)


47760
27583


VALUE
No                                         7000
1.0                                        2863
Yes                                        1842
0.0                                        1433
2.0                                        1292
3.0                                        1146
I have housing                              997
Never smoked tobacco (finding)              986
4.0                                         812
English                                     786
White                                       699
I choose not to answer this question        681
Full-time work                              654
More than high school                       624
Cloudy urine (finding)                      570
Urine smell ammoniacal (finding)            570
Translucent (qualifier value)               570
Brown color (qualifier value)               570
Urine glucose test = ++ (finding)           570
Finding of bilirubin in urine (finding)     570
Name: count, dtype: int64

In [267]:
observations[
    observations["VALUE_NUMERIC"].notna() &
    observations["UNITS"].isna()
]

,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE,VALUE_NUMERIC
47430,2017-08-27T00:20:07Z,a03e9688-2e96-60c9-74cf-f3a3718f729b,a03e9688-2e96-60c9-ccad-d91799805013,exam,271605009,Position of body and posture (observable entity),7.0,NaN,numeric,7.0
47431,2017-10-20T15:11:38Z,a03e9688-2e96-60c9-74cf-f3a3718f729b,a03e9688-2e96-60c9-20f6-5d8ecf68e4b2,exam,285285000,Cobb angle (observable entity),31.7,NaN,numeric,31.7
47432,2017-10-20T15:11:38Z,a03e9688-2e96-60c9-74cf-f3a3718f729b,a03e9688-2e96-60c9-20f6-5d8ecf68e4b2,exam,870537001,Risser sign (finding),2.0,NaN,numeric,2.0


In [268]:
# blood pressure observations
bp = observations[
    observations["DESCRIPTION"]
    .str.contains(
        "blood pressure",
        case=False,
        na=False
    )
]

bp.head(20)



,DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE,VALUE_NUMERIC
4,2017-06-06T16:12:20Z,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-eaff-6e941a8d70ef,vital-signs,8462-4,Diastolic Blood Pressure,83.0,mm[Hg],numeric,83.0
5,2017-06-06T16:12:20Z,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-eaff-6e941a8d70ef,vital-signs,8480-6,Systolic Blood Pressure,125.0,mm[Hg],numeric,125.0
39,2017-04-25T22:34:23Z,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-883e-35bc8b957ff4,vital-signs,8462-4,Diastolic Blood Pressure,69.0,mm[Hg],numeric,69.0
40,2017-04-25T22:34:23Z,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-883e-35bc8b957ff4,vital-signs,8480-6,Systolic Blood Pressure,111.0,mm[Hg],numeric,111.0
70,2018-06-12T16:12:20Z,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-17d7-bd99802e1052,vital-signs,8462-4,Diastolic Blood Pressure,86.0,mm[Hg],numeric,86.0
71,2018-06-12T16:12:20Z,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-17d7-bd99802e1052,vital-signs,8480-6,Systolic Blood Pressure,133.0,mm[Hg],numeric,133.0
114,2018-05-01T22:34:23Z,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-cc03-cbd3dbe3cb43,vital-signs,8462-4,Diastolic Blood Pressure,64.0,mm[Hg],numeric,64.0
115,2018-05-01T22:34:23Z,f9ba83f7-9940-16ae-0854-24bd34bf1843,f9ba83f7-9940-16ae-cc03-cbd3dbe3cb43,vital-signs,8480-6,Systolic Blood Pressure,110.0,mm[Hg],numeric,110.0
150,2019-06-18T16:12:20Z,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-6cd9-60ef5ab899e3,vital-signs,8462-4,Diastolic Blood Pressure,85.0,mm[Hg],numeric,85.0
151,2019-06-18T16:12:20Z,a1733070-046a-4506-bba6-47f32652e9d7,a1733070-046a-4506-6cd9-60ef5ab899e3,vital-signs,8480-6,Systolic Blood Pressure,133.0,mm[Hg],numeric,133.0


In [269]:
# heart rate observations
heart_rate = observations[
    observations["DESCRIPTION"]
    .str.contains(
        "heart rate",
        case=False,
        na=False
    )
]
heart_rate.head(20)
heart_rate['VALUE'].value_counts()

heart_rate[
    ["PATIENT", "DATE", "DESCRIPTION", "VALUE", "UNITS"]
].head(20)

heart_rate["VALUE_NUMERIC"] = pd.to_numeric(
    heart_rate["VALUE"],
    errors="coerce"
)

heart_rate.describe()



,VALUE_NUMERIC
count,1471.000000
mean,80.462339
std,13.892567
min,56.200000
25%,70.000000
50%,80.000000
75%,90.000000
max,195.700000
